# BlastMini Five-Query Demo

这个 notebook 一次性运行 `data/测试序列/` 下的 5 条示例 FASTA，输出结果到 `output/`，并生成一个内嵌 SVG 可视化。


## 1. 准备路径

下面的代码会自动定位项目根目录，并把 `src` 加入 Python import 路径；从项目根目录或 `examples/` 目录打开都可以运行。


In [2]:
from pathlib import Path
from html import escape
from itertools import islice
import csv
import sys

def find_project_root(start):
    start = Path(start).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "src" / "blastmini").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Cannot find BlastMini project root. Please open this notebook inside the blastmini project.")

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

PROJECT_ROOT, OUTPUT_DIR


(PosixPath('/home/lyx/blastmini'), PosixPath('/home/lyx/blastmini/output'))

## 2. 加载五条测试序列


In [3]:
from blastmini.io import read_fasta
from blastmini.core import run_search

QUERY_DIR = PROJECT_ROOT / "data" / "测试序列"
DATABASE_PATH = PROJECT_ROOT / "data" / "uniprot_sprot.fasta"

query_files = sorted(QUERY_DIR.glob("*.fasta"))
all_query_records = []
query_id_to_stem = {}
query_lengths = {}

for path in query_files:
    records = list(read_fasta(str(path)))
    for query_id, sequence in records:
        all_query_records.append((query_id, sequence))
        query_id_to_stem[query_id] = path.stem
        query_lengths[query_id] = len(sequence)

[(query_id_to_stem[qid], qid, len(seq)) for qid, seq in all_query_records]


[('seq_1_glucagon', '1_Glucagon_29aa', 29),
 ('seq_2_ubiquitin', '2_Ubiquitin_76aa', 76),
 ('seq_3_hemoglobin', '3_Hemoglobin_142aa', 142),
 ('seq_4_gfp', '4_GFP_238aa', 238),
 ('seq_5_albumin', '5_Albumin_609aa', 609)]

## 3. 工具函数


In [4]:
RESULT_FIELDS = [
    "query_id",
    "subject_id",
    "score",
    "identity",
    "query_start",
    "query_end",
    "subject_start",
    "subject_end",
    "pvalue",
]

def safe_name(value):
    return "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in str(value))

def write_tsv(rows, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=RESULT_FIELDS, delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)

def group_rows_by_query(rows):
    grouped = {query_id: [] for query_id, _sequence in all_query_records}
    for row in rows:
        grouped.setdefault(row["query_id"], []).append(row)
    return grouped


## 4. 五条序列自比对检查

这一步用五条 query 自己组成一个小数据库，确认每条序列都能跑通。


In [5]:
self_results = run_search(
    query_records=all_query_records,
    db_records=all_query_records,
    k=3,
    top_n=1,
    pvalue_iterations=0,
)

self_output = OUTPUT_DIR / "notebook_all_queries_self.tsv"
write_tsv(self_results, self_output)
self_output, self_results


(PosixPath('/home/lyx/blastmini/output/notebook_all_queries_self.tsv'),
 [{'query_id': '1_Glucagon_29aa',
   'subject_id': '1_Glucagon_29aa',
   'score': 58,
   'identity': '100.00%',
   'query_start': 1,
   'query_end': 29,
   'subject_start': 1,
   'subject_end': 29,
   'pvalue': '1.0000'},
  {'query_id': '2_Ubiquitin_76aa',
   'subject_id': '2_Ubiquitin_76aa',
   'score': 152,
   'identity': '100.00%',
   'query_start': 1,
   'query_end': 76,
   'subject_start': 1,
   'subject_end': 76,
   'pvalue': '1.0000'},
  {'query_id': '3_Hemoglobin_142aa',
   'subject_id': '3_Hemoglobin_142aa',
   'score': 284,
   'identity': '100.00%',
   'query_start': 1,
   'query_end': 142,
   'subject_start': 1,
   'subject_end': 142,
   'pvalue': '1.0000'},
  {'query_id': '4_GFP_238aa',
   'subject_id': '4_GFP_238aa',
   'score': 476,
   'identity': '100.00%',
   'query_start': 1,
   'query_end': 238,
   'subject_start': 1,
   'subject_end': 238,
   'pvalue': '1.0000'},
  {'query_id': '5_Albumin_609aa',

## 5. 五条序列搜索 UniProt 数据库

默认使用 UniProt FASTA 的前 500 条记录做快速演示，五条 query 会一起运行。要跑全库，把 `MAX_DB_RECORDS = 500` 改成 `None`。


In [6]:
MAX_DB_RECORDS = 500
db_stream = read_fasta(str(DATABASE_PATH))
if MAX_DB_RECORDS is not None:
    db_stream = islice(db_stream, MAX_DB_RECORDS)

results = run_search(
    query_records=all_query_records,
    db_records=db_stream,
    k=3,
    top_n=5,
    pvalue_iterations=0,
    random_seed=1,
)

combined_output = OUTPUT_DIR / "notebook_all_queries_results.tsv"
write_tsv(results, combined_output)

per_query_outputs = {}
grouped = group_rows_by_query(results)
for query_id, rows in grouped.items():
    stem = query_id_to_stem.get(query_id, safe_name(query_id))
    output_path = OUTPUT_DIR / f"notebook_{safe_name(stem)}_results.tsv"
    write_tsv(rows, output_path)
    per_query_outputs[query_id] = output_path

combined_output, per_query_outputs


(PosixPath('/home/lyx/blastmini/output/notebook_all_queries_results.tsv'),
 {'1_Glucagon_29aa': PosixPath('/home/lyx/blastmini/output/notebook_seq_1_glucagon_results.tsv'),
  '2_Ubiquitin_76aa': PosixPath('/home/lyx/blastmini/output/notebook_seq_2_ubiquitin_results.tsv'),
  '3_Hemoglobin_142aa': PosixPath('/home/lyx/blastmini/output/notebook_seq_3_hemoglobin_results.tsv'),
  '4_GFP_238aa': PosixPath('/home/lyx/blastmini/output/notebook_seq_4_gfp_results.tsv'),
  '5_Albumin_609aa': PosixPath('/home/lyx/blastmini/output/notebook_seq_5_albumin_results.tsv')})

## 6. 查看合并结果


In [7]:
with combined_output.open("r", encoding="utf-8") as handle:
    for line in list(handle)[:12]:
        print(line.rstrip())


query_id	subject_id	score	identity	query_start	query_end	subject_start	subject_end	pvalue
1_Glucagon_29aa	sp|Q6GZR6|059L_FRG3G	10	75.00%	10	17	287	294	1.0000
1_Glucagon_29aa	sp|Q6GZW4|011R_FRG3G	10	75.00%	5	12	26	33	1.0000
1_Glucagon_29aa	sp|P0DO87|11H_STRNX	10	100.00%	14	18	262	266	1.0000
1_Glucagon_29aa	sp|Q6GZV8|017L_FRG3G	10	100.00%	8	12	5	9	1.0000
1_Glucagon_29aa	sp|P13744|11SB_CUCMA	9	83.33%	18	23	54	59	1.0000
2_Ubiquitin_76aa	sp|O55704|060L_IIV6	12	66.67%	12	23	201	212	1.0000
2_Ubiquitin_76aa	sp|Q99002|1433_TRIHA	11	70.00%	23	32	88	97	1.0000
2_Ubiquitin_76aa	sp|Q6GZT5|041R_FRG3G	10	75.00%	4	11	786	793	1.0000
2_Ubiquitin_76aa	sp|Q6GZX0|005R_FRG3G	10	75.00%	4	11	45	52	1.0000
2_Ubiquitin_76aa	sp|O55735|121R_IIV6	10	100.00%	23	27	36	40	1.0000
3_Hemoglobin_142aa	sp|Q6GZR8|057R_FRG3G	12	66.67%	11	22	297	308	1.0000


## 7. 可视化 Top Hits

下面的图按 query 分组展示 top hits：左侧是 subject，中央条形表示 score，右侧小轨道表示命中覆盖的 query 区间。颜色表示 identity。


In [8]:
def identity_value(row):
    return float(str(row["identity"]).rstrip("%"))

def color_for_identity(identity):
    if identity >= 95:
        return "#2a9d8f"
    if identity >= 80:
        return "#4d96ff"
    if identity >= 65:
        return "#e9c46a"
    return "#e76f51"

def shorten(value, width=44):
    value = str(value)
    return value if len(value) <= width else value[: width - 1] + "…"

def render_hits_svg(rows):
    grouped = group_rows_by_query(rows)
    max_score = max([int(row["score"]) for row in rows] or [1])
    width = 1080
    left = 230
    bar_x = 470
    bar_w = 260
    track_x = 820
    track_w = 180
    row_h = 34
    y = 42
    parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="10" font-family="Arial, sans-serif">',
        '<rect x="0" y="0" width="100%" height="100%" fill="#ffffff"/>',
        '<text x="24" y="28" font-size="20" font-weight="700" fill="#1f2937">BlastMini top hits for five demo queries</text>',
        '<text x="470" y="28" font-size="12" fill="#6b7280">score</text>',
        '<text x="820" y="28" font-size="12" fill="#6b7280">query interval</text>',
    ]

    for query_id, sequence in all_query_records:
        query_rows = grouped.get(query_id, [])
        query_label = f"{query_id_to_stem.get(query_id, query_id)} · {query_id} · {len(sequence)} aa"
        parts.append(f'<text x="24" y="{y}" font-size="14" font-weight="700" fill="#111827">{escape(query_label)}</text>')
        y += 18

        if not query_rows:
            parts.append(f'<text x="44" y="{y}" font-size="12" fill="#9ca3af">No seed hits in the selected database slice</text>')
            y += row_h
            continue

        for row in query_rows:
            score = int(row["score"])
            identity = identity_value(row)
            color = color_for_identity(identity)
            bar_len = max(3, int(score / max_score * bar_w))
            q_len = max(1, query_lengths.get(query_id, int(row["query_end"])))
            q_start = int(row["query_start"])
            q_end = int(row["query_end"])
            hit_x = track_x + int((q_start - 1) / q_len * track_w)
            hit_w = max(3, int((q_end - q_start + 1) / q_len * track_w))
            subject = escape(shorten(row["subject_id"]))
            parts.extend([
                f'<text x="44" y="{y}" font-size="12" fill="#374151">{subject}</text>',
                f'<rect x="{bar_x}" y="{y - 12}" width="{bar_len}" height="14" rx="3" fill="{color}"/>',
                f'<text x="{bar_x + bar_w + 12}" y="{y}" font-size="12" fill="#374151">{score}</text>',
                f'<text x="{bar_x + bar_w + 54}" y="{y}" font-size="12" fill="#374151">{row["identity"]}</text>',
                f'<line x1="{track_x}" y1="{y - 5}" x2="{track_x + track_w}" y2="{y - 5}" stroke="#d1d5db" stroke-width="4" stroke-linecap="round"/>',
                f'<rect x="{hit_x}" y="{y - 10}" width="{hit_w}" height="10" rx="2" fill="{color}"/>',
                f'<text x="{track_x + track_w + 12}" y="{y}" font-size="12" fill="#6b7280">{q_start}-{q_end}</text>',
            ])
            y += row_h
        y += 10

    parts.append('</svg>')
    svg = "\n".join(parts)
    svg = svg.replace('height="10"', f'height="{y + 20}"', 1)
    return svg

svg = render_hits_svg(results)
visualization_path = OUTPUT_DIR / "notebook_blastmini_visualization.html"
visualization_path.write_text(
    "<!doctype html><meta charset='utf-8'><title>BlastMini Demo Visualization</title>" + svg,
    encoding="utf-8",
)

try:
    from IPython.display import HTML, display
    display(HTML(svg))
except Exception:
    print(f"Visualization written to {visualization_path}")

visualization_path


PosixPath('/home/lyx/blastmini/output/notebook_blastmini_visualization.html')

## 8. 输出文件


In [9]:
for path in sorted(OUTPUT_DIR.glob("notebook_*")):
    print(path.relative_to(PROJECT_ROOT))


output/notebook_all_queries_results.tsv
output/notebook_all_queries_self.tsv
output/notebook_blastmini_visualization.html
output/notebook_seq_1_glucagon_results.tsv
output/notebook_seq_2_ubiquitin_results.tsv
output/notebook_seq_3_hemoglobin_results.tsv
output/notebook_seq_4_gfp_results.tsv
output/notebook_seq_5_albumin_results.tsv
